## 02c — RAG Strategy 3: Dual-Index Retrieval (Separate Context and Holdings Indexes)

Builds two separate FAISS indexes over Legal-BERT embeddings (`nlpaueb/legal-bert-base-uncased`): one for context chunks and one for holding chunks from 1,000 CaseHOLD training examples (chunk size 500 chars, overlap 50, yielding 1,554 total vectors). At inference time, the top-3 chunks from each index are combined and passed to `mistralai/Mistral-7B-Instruct-v0.2`.

**Note on inference metrics:** The inference loop was interrupted by the Colab session. Evaluation metrics (ROUGE-L and Legal-BERT cosine similarity) in the first evaluation cell are computed on a partial sample of approximately 41–49 of the 3,900 validation examples — not the full validation set.

Starts from HERE!!

In [25]:
# -------------------------
# 2. Prepare RAG retrieval (Option 3: separate indexes for context & holdings)
# -------------------------

# Split and chunk contexts
context_docs = [row["context"] for row in train_dataset.select(range(1000))]
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)

context_chunks = []
for i, doc in enumerate(context_docs):
    for chunk in splitter.split_text(doc):
        context_chunks.append({"id": i, "text": chunk})

# Chunk holdings separately
holding_docs = [row["endings"][row["label"]] for row in train_dataset.select(range(1000))]
holding_chunks = []
for i, doc in enumerate(holding_docs):
    for chunk in splitter.split_text(doc):
        holding_chunks.append({"id": i, "text": chunk})

In [26]:
# -------------------------
# 3. Legal-BERT embeddings
# -------------------------
from transformers import AutoTokenizer, AutoModel

tokenizer_bert = AutoTokenizer.from_pretrained("nlpaueb/legal-bert-base-uncased")
model_bert = AutoModel.from_pretrained("nlpaueb/legal-bert-base-uncased")

def get_bert_embeddings(text_list, batch_size=256):
    all_embeddings = []
    for i in range(0, len(text_list), batch_size):
        batch = text_list[i:i+batch_size]
        inputs = tokenizer_bert(batch, return_tensors="pt", padding=True, truncation=True, max_length=512).to(model_bert.device)
        with torch.no_grad():
            outputs = model_bert(**inputs)
            emb = outputs.last_hidden_state.mean(dim=1).cpu().numpy()
            all_embeddings.append(emb)
        torch.cuda.empty_cache()
    return np.vstack(all_embeddings)

# Embed context and holdings
import numpy as np
context_texts = [chunk["text"] for chunk in context_chunks]
holding_texts = [chunk["text"] for chunk in holding_chunks]

context_embeddings = get_bert_embeddings(context_texts)
holding_embeddings = get_bert_embeddings(holding_texts)

In [27]:

# -------------------------
# 4. FAISS indexes
# -------------------------
dim = context_embeddings.shape[1]
context_index = faiss.IndexFlatL2(dim)
context_index.add(context_embeddings)

holding_index = faiss.IndexFlatL2(dim)
holding_index.add(holding_embeddings)

In [28]:

# -------------------------
# 6. RAG inference functions
# -------------------------
def retrieve_dual(query, top_k=3):
    # encode query with Legal-BERT
    inputs = tokenizer_bert(query, return_tensors="pt", truncation=True, padding=True, max_length=512).to(model_bert.device)
    with torch.no_grad():
        query_emb = model_bert(**inputs).last_hidden_state.mean(dim=1).cpu().numpy()

    # search context
    d1, i1 = context_index.search(query_emb, top_k)
    context_ret = [context_chunks[idx]["text"] for idx in i1[0]]

    # search holdings
    d2, i2 = holding_index.search(query_emb, top_k)
    holding_ret = [holding_chunks[idx]["text"] for idx in i2[0]]

    return context_ret, holding_ret

def ask_mistral(context_chunks, holding_chunks, query, max_new_tokens=256):
    prompt = f"""
You are a legal assistant.
Use ONLY the retrieved context and holdings below to answer the question.
If the holding is present, copy it verbatim.
Do NOT paraphrase, invent, or summarize.

Contexts:
{chr(10).join(context_chunks)}

Holdings:
{chr(10).join(holding_chunks)}

Question: {query}
Answer:
"""
    inputs = tokenizer_mistral(prompt, return_tensors="pt").to(model_mistral.device)
    with torch.no_grad():
        outputs = model_mistral.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.1,
            do_sample=False
        )
    answer = tokenizer_mistral.decode(outputs[0], skip_special_tokens=True)
    return answer.replace(prompt, "").strip()

def run_rag_inference(query, top_k=3):
    context_ret, holding_ret = retrieve_dual(query, top_k)
    answer = ask_mistral(context_ret, holding_ret, query)
    return answer, context_ret, holding_ret

In [29]:
# -------------------------
# 7. Run inference on validation set
# -------------------------
results = []

for i, example in enumerate(test_dataset):
    gold = example["endings"][example["label"]]
    query = "What is the holding?"
    answer, ctx_used, hold_used = run_rag_inference(query)

    results.append({
        "id": str(i),
        "context": example["context"],
        "gold_holding": gold,
        "predicted_answer": answer
    })
    if i % 10 == 0:
        print(f"Processed {i} examples...")

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Processed 0 examples...


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Processed 10 examples...


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


...

Processed 30 examples...


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Processed 40 examples...


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


KeyboardInterrupt: 

**Note on results:** The inference loop was interrupted before completion. Metrics below are computed on a partial sample of approximately 41–49 of 3,900 validation examples. Results should be read as preliminary, not full-dataset evaluation.

In [30]:
# -------------------------
# 8. Evaluation: ROUGE-L + Cosine Similarity with Legal-BERT
# -------------------------
rouge = evaluate.load("rouge")

rouge_scores, cosine_scores = [], []

for row in results:
    pred = row["predicted_answer"]
    gold = row["gold_holding"]

    # ROUGE-L
    rouge_l = rouge.compute(predictions=[pred], references=[gold], rouge_types=["rougeL"])["rougeL"]
    rouge_scores.append(rouge_l)

    # Cosine similarity
    emb = get_bert_embeddings([pred, gold])
    cos_sim = cosine_similarity([emb[0]], [emb[1]])[0][0]
    cosine_scores.append(cos_sim)

# Save metrics CSV
df_metrics = pd.DataFrame({
    "id": [row["id"] for row in results],
    "gold": [row["gold_holding"] for row in results],
    "predicted": [row["predicted_answer"] for row in results],
    "rougeL": rouge_scores,
    "cosine": cosine_scores
})
df_metrics.to_csv("pipeline1_rag_option3_mistral_legalbert_eval.csv", index=False)
print("✅ Avg ROUGE-L:", sum(rouge_scores)/len(rouge_scores))
print("✅ Avg Cosine:", sum(cosine_scores)/len(cosine_scores))

✅ Avg ROUGE-L: 0.08033598246653899
✅ Avg Cosine: 0.8491007075423286


option 3 is above


In [1]:
import json
import faiss
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd
import random
import subprocess


In [2]:
from rouge_score import rouge_scorer
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

In [3]:
from huggingface_hub import notebook_login # hf_REDACTED
notebook_login()

In [4]:
from datasets import load_dataset

dataset = load_dataset("coastalcph/lex_glue", "case_hold")
train_dataset = dataset["train"]
test_dataset = dataset["validation"]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [5]:
# 2. Prepare RAG retrieval (context + gold holding together)
# -------------------------
from langchain.text_splitter import RecursiveCharacterTextSplitter

docs = [row["context"] + " " + row["endings"][row["label"]] for row in train_dataset]

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = []
for i, doc in enumerate(docs):
    for chunk in splitter.split_text(doc):
        chunks.append({"id": i, "text": chunk})

chunks_texts = [chunk["text"] for chunk in chunks]

In [9]:
from transformers import AutoTokenizer, AutoModel
import torch
#import faiss
import numpy as np

# 3. Legal-BERT embeddings for retrieval
# -------------------------
tokenizer_bert = AutoTokenizer.from_pretrained("nlpaueb/legal-bert-base-uncased")
model_bert = AutoModel.from_pretrained("nlpaueb/legal-bert-base-uncased").to("cuda")

def get_bert_embeddings(text_list, batch_size=256):
    all_embeddings = []
    for i in range(0, len(text_list), batch_size):
        batch = text_list[i:i+batch_size]
        inputs = tokenizer_bert(batch, return_tensors="pt", padding=True, truncation=True, max_length=512).to("cuda")
        with torch.no_grad():
            outputs = model_bert(**inputs)
            emb = outputs.last_hidden_state.mean(dim=1).cpu().numpy()
            all_embeddings.append(emb)
        torch.cuda.empty_cache()
    return np.vstack(all_embeddings)

embeddings = get_bert_embeddings(chunks_texts)



In [10]:
# -------------------------
# 4. FAISS index
# -------------------------
dim = embeddings.shape[1]
index = faiss.IndexFlatL2(dim)
index.add(embeddings)

In [16]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# -------------------------
# 1. Load Mistral-7B-Instruct-v0.2
# -------------------------
model_name = "mistralai/Mistral-7B-Instruct-v0.2"

tokenizer_mistral = AutoTokenizer.from_pretrained(model_name)
model_mistral = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,  # saves memory
    device_map="auto"           # auto-distributes across GPU if available
)

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [17]:

# -------------------------
# 6. RAG inference functions
# -------------------------
def retrieve(query, top_k=3):
    inputs = tokenizer_bert(query, return_tensors="pt", truncation=True, padding=True, max_length=512).to("cuda")
    with torch.no_grad():
        query_emb = model_bert(**inputs).last_hidden_state.mean(dim=1).cpu().numpy()
    distances, indices = index.search(query_emb, top_k)
    return [chunks[i]["text"] for i in indices[0]]

def ask_mistral(context_chunks, query, max_new_tokens=256):
    prompt = f"""
You are a legal assistant.
Use ONLY the retrieved context below to answer the question.
If the holding is present, copy it verbatim.
Do NOT paraphrase, invent, or summarize.

Retrieved Context:
{chr(10).join(context_chunks)}

Question: {query}
Answer:
"""
    inputs = tokenizer_mistral(prompt, return_tensors="pt").to(model_mistral.device)
    with torch.no_grad():
        outputs = model_mistral.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.1,
            do_sample=False
        )
    answer = tokenizer_mistral.decode(outputs[0], skip_special_tokens=True)
    return answer.replace(prompt, "").strip()

def run_rag_inference(query, top_k=3):
    retrieved = retrieve(query, top_k)
    answer = ask_mistral(retrieved, query)
    return answer, retrieved

In [18]:
# -------------------------
# 7. Run inference on validation set
# -------------------------
results = []

for i, example in enumerate(test_dataset):
    gold = example["endings"][example["label"]]
    query = "What is the holding?"

    answer, docs_used = run_rag_inference(query)
    results.append({
        "id": str(i),
        "context": example["context"],
        "gold_holding": gold,
        "predicted_answer": answer
    })

    if i % 10 == 0:
        print(f"Processed {i} examples...")

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Processed 0 examples...


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Processed 10 examples...


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


...

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Processed 30 examples...


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Processed 40 examples...


KeyboardInterrupt: 

In [22]:
import evaluate
# -------------------------
# 8. Evaluation: ROUGE-L + Cosine Similarity using Legal-BERT
# -------------------------
rouge = evaluate.load("rouge")

rouge_scores, cosine_scores = [], []

for row in results:
    pred = row["predicted_answer"]
    gold = row["gold_holding"]

    # ROUGE-L
    rouge_l = rouge.compute(predictions=[pred], references=[gold], rouge_types=["rougeL"])["rougeL"]
    rouge_scores.append(rouge_l)

    # Cosine similarity
    emb = get_bert_embeddings([pred, gold])
    cos_sim = cosine_similarity([emb[0]], [emb[1]])[0][0]
    cosine_scores.append(cos_sim)

# Save metrics CSV
df_metrics = pd.DataFrame({
    "id": [row["id"] for row in results],
    "gold": [row["gold_holding"] for row in results],
    "predicted": [row["predicted_answer"] for row in results],
    "rougeL": rouge_scores,
    "cosine": cosine_scores
})
#df_metrics.to_csv("pipeline1_rag_option2_mistral_legalbert_eval.csv", index=False)
print("✅ Avg ROUGE-L:", sum(rouge_scores)/len(rouge_scores))
print("✅ Avg Cosine:", sum(cosine_scores)/len(cosine_scores))

✅ Avg ROUGE-L: 0.15598328333121342
✅ Avg Cosine: 0.8652554416074986


OPTION 2

In [21]:
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.5 MB/s eta 0:00:00


In [23]:
from transformers import AutoTokenizer, AutoModel
import torch
import faiss
import numpy as np

# -------------------------
# 3. Create embeddings and FAISS index using Legal-BERT
# -------------------------

def get_legal_bert_embedding(text):
    inputs = tokenizer_legal(text, return_tensors="pt", truncation=True, padding=True, max_length=512).to(device)
    with torch.no_grad():
        outputs = model_legal(**inputs)
    # Mean pooling over token embeddings
    embedding = outputs.last_hidden_state.mean(dim=1)
    return embedding.squeeze().cpu().numpy()

# Generate embeddings for all chunks
embeddings = []
for chunk_text in chunks_texts:
    emb = get_legal_bert_embedding(chunk_text)
    embeddings.append(emb)
embeddings = np.array(embeddings)

# Build FAISS index
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

print(f"✅ FAISS index created with {len(embeddings)} embeddings of dimension {dimension}")







✅ FAISS index created with 1554 embeddings of dimension 768


In [26]:

# -------------------------
# 5. Define inference functions
# -------------------------
def retrieve(query, top_k=3):
    query_vec = embedder.encode([query], convert_to_numpy=True)
    distances, indices = index.search(query_vec, top_k)
    return [chunks[i]["text"] for i in indices[0]]

def ask_mistral(context, query, max_new_tokens=256):
    prompt = f"""
You are a legal assistant.
Use ONLY the retrieved context below to answer the question.
If the holding is present in the context, copy it verbatim.
Do NOT paraphrase, invent, or summarize.

Retrieved Context:
{context}

Question: {query}
Answer:
"""
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=0.1,
        do_sample=False
    )
    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return answer.replace(prompt, "").strip()

def run_rag_inference(query):
    retrieved = retrieve(query, top_k=3)
    context = "\n".join(retrieved)
    answer = ask_mistral(context, query)
    return answer, retrieved

In [29]:



# -------------------------
# 3. Run inference on test set
# -------------------------
results = []

for i, example in enumerate(test_dataset):
    gold = example["endings"][example["label"]]
    query = "What is the holding?"

    answer, docs_used = run_rag_inference(query=query)

    results.append({
        "id": str(i),
        "context": example["context"],
        "gold_holding": gold,
        "predicted_answer": answer
    })

    if i % 10 == 0:
        print(f"Processed {i} examples...")


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


OutOfMemoryError: CUDA out of memory. Tried to allocate 172.00 MiB. GPU 0 has a total capacity of 14.74 GiB of which 104.12 MiB is free. Process 14942 has 14.64 GiB memory in use. Of the allocated memory 14.22 GiB is allocated by PyTorch, and 295.88 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [28]:
import torch
torch.cuda.empty_cache()